# 🐝 Spark SQL & Hive Data Warehousing Analytics

**Relational Querying over Distributed HDFS Storage**  
This notebook demonstrates how to execute distributed SQL queries over datasets stored in HDFS and Hive Warehouse directories using Apache Spark SQL.

In [ ]:
# Step 1: Start SparkSession with SQL & Catalog support
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark-SQL-Hive-Analytics") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.sql.warehouse.dir", "hdfs://localhost:9000/user/hive/warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("⚡ Spark SQL Session Active!")

In [ ]:
# Step 2: Register Temporary and Managed Tables
sample_sales = [
    (101, "Product-A", "Electronics", 1200.0, 3, "Cairo", "2026-05-01"),
    (102, "Product-B", "Home", 450.0, 10, "Alexandria", "2026-05-02"),
    (103, "Product-C", "Electronics", 890.0, 5, "Giza", "2026-05-03"),
    (104, "Product-D", "Books", 65.0, 42, "Cairo", "2026-05-03"),
    (105, "Product-E", "Home", 310.0, 8, "Cairo", "2026-05-04"),
    (106, "Product-F", "Electronics", 2100.0, 2, "Alexandria", "2026-05-05")
]

cols = ["sale_id", "product_name", "category", "unit_price", "quantity", "city", "sale_date"]
df_sales = spark.createDataFrame(sample_sales, cols)
df_sales.createOrReplaceTempView("sales_orders")

print("=== Registered Temp View 'sales_orders' ===")
spark.sql("SELECT * FROM sales_orders LIMIT 3").show()

In [ ]:
# Step 3: Complex Analytical Query with Window Functions
sql_query = """
SELECT 
    category,
    product_name,
    city,
    (unit_price * quantity) AS total_revenue,
    DENSE_RANK() OVER (PARTITION BY category ORDER BY (unit_price * quantity) DESC) as category_rank
FROM sales_orders
ORDER BY category, category_rank
"""

print("=== Analytical Ranking by Revenue ===")
spark.sql(sql_query).show()

In [ ]:
# Step 4: Write Aggregated Revenue Lake Table to HDFS
hdfs_dest = "hdfs://localhost:9000/data/revenue_by_category.parquet"
spark.sql("""
    SELECT 
        category, 
        COUNT(*) AS order_count, 
        ROUND(SUM(unit_price * quantity), 2) AS category_revenue
    FROM sales_orders 
    GROUP BY category
""").write.mode("overwrite").parquet(hdfs_dest)

print(f"🎉 Successfully persisted aggregated revenue table to HDFS: {hdfs_dest}")
spark.stop()